In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from scipy.stats import ttest_ind
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR_CANDIDATES = (Path("anticp2"), Path("."))
DATA_DIR = next(
    (path for path in DATA_DIR_CANDIDATES if (path / "anticp2_datasets_with_features.csv").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Could not locate the anticp2 data directory.")

SPLIT_DIR_CANDIDATES = (DATA_DIR, Path("datasets/anticp2"), Path("../datasets/anticp2"))
SPLIT_DIR = next(
    (path for path in SPLIT_DIR_CANDIDATES if (path / "internal_776.csv").is_file()),
    None,
)
if SPLIT_DIR is None:
    raise FileNotFoundError("Could not locate the anticp2 train/validation files.")

combined_dataset = pd.read_csv(DATA_DIR / "anticp2_datasets_with_features.csv")


## Standardization and statistical feature selection

In [ ]:
features_to_scale = combined_dataset.drop(columns=["data", "label"])
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_to_scale)
scaled_df = pd.DataFrame(scaled_features, columns=features_to_scale.columns)
scaled_df["data"] = combined_dataset["data"].values
scaled_df["label"] = combined_dataset["label"].values
combined_dataset = scaled_df[["data", "label", *features_to_scale.columns]]

group_0 = combined_dataset.loc[combined_dataset["label"] == 0].drop(columns=["data", "label"])
group_1 = combined_dataset.loc[combined_dataset["label"] == 1].drop(columns=["data", "label"])
t_stats, p_values = ttest_ind(group_0, group_1, equal_var=False)
ttest_results = pd.DataFrame(
    {"feature": group_0.columns, "t_stat": t_stats, "p_value": p_values}
)

nan_features = ttest_results.loc[ttest_results["p_value"].isna(), "feature"]
cols_to_drop = [column for column in nan_features if column in combined_dataset.columns]
combined_dataset = combined_dataset.drop(columns=cols_to_drop)
ttest_results = ttest_results.dropna()

p_values_df = ttest_results[["feature", "p_value"]].copy()
_, q_values, _, _ = multipletests(p_values_df["p_value"], method="fdr_bh")
p_values_df["q_value"] = q_values
significant_features = p_values_df.loc[p_values_df["q_value"] < 0.01, "feature"].tolist()
cleaned_data = combined_dataset[["data", "label", *significant_features]]


## Train-validation split preparation

In [ ]:
anticp2_train = pd.concat(
    [
        pd.read_csv(SPLIT_DIR / "internal_776.csv"),
        pd.read_csv(SPLIT_DIR / "internalNon_776.csv"),
    ],
    ignore_index=True,
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
anticp2_validation = pd.concat(
    [
        pd.read_csv(SPLIT_DIR / "validation_194.csv"),
        pd.read_csv(SPLIT_DIR / "validationNon_194.csv"),
    ],
    ignore_index=True,
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

anticp2_train_df = cleaned_data[
    cleaned_data["data"].isin(anticp2_train["data"])
].copy()
anticp2_validation_df = cleaned_data[
    cleaned_data["data"].isin(anticp2_validation["data"])
].copy()


## CatBoost model

In [ ]:
X_train = anticp2_train_df.drop(columns=["data", "label"])
y_train = anticp2_train_df["label"]
X_validation = anticp2_validation_df.drop(columns=["data", "label"])
y_validation = anticp2_validation_df["label"]

def specificity_score(y_true, y_pred):
    tn, fp, _, _ = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    random_state=RANDOM_STATE,
    verbose=0,
    allow_writing_files=False,
)
scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "mcc": make_scorer(matthews_corrcoef),
    "specificity": make_scorer(specificity_score),
}
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1,
)

def mean_std(scores):
    return f"{np.mean(scores):.4f} +/- {np.std(scores):.4f}"

print(f"Train size: {len(X_train)}, Validation size: {len(X_validation)}")
print("\n10-Fold CV Results (Train Set - CatBoost):")
for metric, label in (
    ("accuracy", "Accuracy"),
    ("f1", "F1 Score"),
    ("recall", "Sensitivity"),
    ("specificity", "Specificity"),
    ("precision", "Precision"),
    ("roc_auc", "AUC (ROC)"),
    ("mcc", "MCC"),
):
    print(f"{label:<13}: {mean_std(cv_results[f'test_{metric}'])}")

model.fit(X_train, y_train)
y_pred = model.predict(X_validation)
y_probability = model.predict_proba(X_validation)[:, 1]
tn, fp, fn, tp = confusion_matrix(y_validation, y_pred).ravel()

print("\nValidation Set Results (CatBoost):")
print(f"Accuracy     : {accuracy_score(y_validation, y_pred):.4f}")
print(f"F1 Score     : {f1_score(y_validation, y_pred):.4f}")
print(f"Sensitivity  : {recall_score(y_validation, y_pred):.4f}")
print(f"Specificity  : {tn / (tn + fp):.4f}")
print(f"Precision    : {precision_score(y_validation, y_pred):.4f}")
print(f"AUC (ROC)    : {roc_auc_score(y_validation, y_probability):.4f}")
print(f"MCC          : {matthews_corrcoef(y_validation, y_pred):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_validation, y_pred)}")
print(f"\nClassification Report:\n{classification_report(y_validation, y_pred)}")
